In [0]:
%sql
create catalog if not exists autoloader;
create schema if not exists autoloader.bronze;
create schema if not exists autoloader.silver;
create schema if not exists autoloader.gold;
use catalog autoloader;

In [0]:
# Step 2: Define S3 file paths
inputfile = "s3://streamautoloader/input"
schemalocation = "s3://streamautoloader/schema"
checkpointlocation = "s3://streamautoloader/checkpoint"
outputlocation = "s3://streamautoloader/output"

In [0]:
# Step 3: Import required libraries
from delta.tables import *
from pyspark.sql.functions import col, lit, unix_timestamp, from_unixtime, to_timestamp, to_date, current_timestamp, when, count, isnan
from pyspark.sql.types import *
from pyspark.sql import *
from pyspark.sql.window import Window
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('autoloader_pipeline')
logger.info('Libraries imported successfully')

In [0]:
# Step 4: Read CSV files from S3 using Autoloader
inputfile = "s3://streamautoloader/input/"
schemalocation = "s3://streamautoloader/schema/"
checkpointlocation = "s3://streamautoloader/checkpoint/"
outputlocation = "s3://streamautoloader/output/"

try:
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schemalocation)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("header", "true")
        .load(inputfile)
    )
    logger.info('Streaming DataFrame created successfully')
except Exception as e:
    logger.error(f'Error creating streaming DataFrame: {str(e)}')
    raise

In [0]:
# Step 5: Transformation Function
def transform_dataframe(df):
    """
    Apply transformations to the ingested DataFrame.
    - Cleans column names
    - Adds ingestion timestamp
    - Adds source file tracking
    """
    try:
        # Clean column names: lowercase and replace spaces with underscores
        for col_name in df.columns:
            df = df.withColumnRenamed(col_name, col_name.lower().replace(' ', '_').replace('-', '_'))

        # Add ingestion timestamp
        df = df.withColumn('ingestion_timestamp', current_timestamp())

        # Add source file name
        df = df.withColumn('source_file', col('_metadata.file_path'))

        logger.info('Transformations applied successfully')
        return df
    except Exception as e:
        logger.error(f'Error in transformation: {str(e)}')
        raise

# Apply transformation
df_transformed = transform_dataframe(df)
logger.info('DataFrame transformed successfully')

In [0]:
# Step 5: Transformation Function
def transform_dataframe(df):
    """
    Apply transformations to the ingested DataFrame.
    - Cleans column names
    - Adds ingestion timestamp
    - Adds source file tracking
    """
    try:
        # Clean column names: lowercase and replace spaces with underscores
        for col_name in df.columns:
            df = df.withColumnRenamed(col_name, col_name.lower().replace(' ', '_').replace('-', '_'))

        # Add ingestion timestamp
        df = df.withColumn('ingestion_timestamp', current_timestamp())

        # Add source file name
        df = df.withColumn('source_file', col('_metadata.file_path'))

        logger.info('Transformations applied successfully')
        return df
    except Exception as e:
        logger.error(f'Error in transformation: {str(e)}')
        raise

# Apply transformation
df_transformed = transform_dataframe(df)
logger.info('DataFrame transformed successfully')

In [0]:
# Step 6: Data Validation Rules
def validate_dataframe(df, key_columns=None):
    """
    Validate the DataFrame before writing.
    Checks: nulls, duplicates, empty dataset.
    Returns: (is_valid: bool, report: dict)
    """
    report = {}
    is_valid = True

    try:
        # 1. Row count check
        row_count = df.count()
        report['total_rows'] = row_count
        if row_count == 0:
            logger.warning('Validation FAILED: DataFrame is empty!')
            report['empty_check'] = 'FAILED - No data found'
            is_valid = False
        else:
            report['empty_check'] = f'PASSED - {row_count} rows found'
            logger.info(f'Row count check passed: {row_count} rows')

        # 2. Null check on key columns
        if key_columns:
            for col_name in key_columns:
                if col_name in df.columns:
                    null_count = df.filter(col(col_name).isNull() | isnan(col(col_name))).count()
                    if null_count > 0:
                        report[f'null_check_{col_name}'] = f'FAILED - {null_count} nulls found'
                        logger.warning(f'Null check FAILED for {col_name}: {null_count} nulls')
                        is_valid = False
                    else:
                        report[f'null_check_{col_name}'] = 'PASSED'
                        logger.info(f'Null check passed for {col_name}')

        # 3. Duplicate check
        if key_columns:
            total = df.count()
            distinct = df.dropDuplicates(key_columns).count()
            dup_count = total - distinct
            if dup_count > 0:
                report['duplicate_check'] = f'WARNING - {dup_count} duplicate rows found'
                logger.warning(f'Duplicate check: {dup_count} duplicates found')
            else:
                report['duplicate_check'] = 'PASSED - No duplicates'
                logger.info('Duplicate check passed')

        # Print validation report
        print('\n===== VALIDATION REPORT =====')
        for key, value in report.items():
            print(f'  {key}: {value}')
        print(f'  overall_status: {"PASSED" if is_valid else "FAILED"}')
        print('=============================')

        return is_valid, report

    except Exception as e:
        logger.error(f'Error during validation: {str(e)}')
        raise

In [0]:
# Step 7: Write to Delta Table with Error Handling
def write_to_delta(df, table_name, checkpoint_location):
    """
    Write streaming DataFrame to Delta table with error handling.
    """
    try:
        query = (
            df.writeStream
            .format('delta')
            .option('checkpointLocation', checkpoint_location)
            .trigger(availableNow=True)
            .outputMode('append')
            .toTable(table_name)
        )
        logger.info(f'Successfully writing to {table_name}')
        return query
    except Exception as e:
        logger.error(f'Error writing to Delta table {table_name}: {str(e)}')
        raise

# Execute write
try:
    stream_query = write_to_delta(
        df_transformed,
        'autoloader.bronze.sales',
        checkpointlocation
    )
    logger.info('Pipeline completed successfully!')
except Exception as e:
    logger.error(f'Pipeline failed: {str(e)}')
    raise

In [0]:
from pyspark.sql.functions import sum

gold_df = (
    silver_df
    .groupBy("city", "order_date")
    .agg(sum("amount").alias("total_sales"))
)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS autoloader.gold")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("autoloader.gold.sales_summary")

In [0]:
%sql
SELECT * FROM autoloader.gold.sales_summary;

In [0]:
display(dbutils.fs.ls("s3://streamautoloader/input/"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("city").orderBy(desc("total_sales"))

top_products = (
    gold_df
    .withColumn("rank", row_number().over(window))
    .filter("rank = 1")
)

In [0]:
top_products.show()

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS autoloader.gold.customer_scd (
    order_id STRING,
    product STRING,
    amount INT,
    city STRING,
    order_date DATE,
    is_current STRING,
    start_date DATE,
    end_date DATE
)
USING DELTA
""")

In [0]:
source_df = spark.read.table("autoloader.silver.sales_clean")

In [0]:
from pyspark.sql.functions import current_date, lit

source_df = (
    source_df
    .withColumn("is_current", lit("Y"))
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast("date"))
)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "autoloader.gold.customer_scd")

(
    target.alias("t")
    .merge(
        source_df.alias("s"),
        "t.order_id = s.order_id AND t.is_current = 'Y'"
    )
    .whenMatchedUpdate(
        condition="t.city <> s.city",
        set={
            "is_current": "'N'",
            "end_date": "current_date()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "product": "s.product",
            "amount": "s.amount",
            "city": "s.city",
            "order_date": "s.order_date",
            "is_current": "s.is_current",
            "start_date": "s.start_date",
            "end_date": "s.end_date"
        }
    )
    .execute()
)

In [0]:
%sql
SELECT * FROM autoloader.gold.customer_scd
ORDER BY order_id, start_date;

In [0]:
silver_stream_df = (
    spark.readStream
    .table("autoloader.silver.sales_clean")
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date, lit

def scd2_merge(microBatchDF, batchId):

    # Add SCD columns
    df = (
        microBatchDF
        .withColumn("is_current", lit("Y"))
        .withColumn("start_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
    )

    target = DeltaTable.forName(spark, "autoloader.gold.customer_scd")

    (
        target.alias("t")
        .merge(
            df.alias("s"),
            "t.order_id = s.order_id AND t.is_current = 'Y'"
        )
        .whenMatchedUpdate(
            condition="t.city <> s.city",
            set={
                "is_current": "'N'",
                "end_date": "current_date()"
            }
        )
        .whenNotMatchedInsert(
            values={
                "order_id": "s.order_id",
                "product": "s.product",
                "amount": "s.amount",
                "city": "s.city",
                "order_date": "s.order_date",
                "is_current": "s.is_current",
                "start_date": "s.start_date",
                "end_date": "s.end_date"
            }
        )
        .execute()
    )

In [0]:
(
    silver_stream_df.writeStream
    .foreachBatch(scd2_merge)
    .option("checkpointLocation", "s3://streamautoloader/scd_checkpoint/")
    .trigger(availableNow=True)
    .start()
)

In [0]:
%sql
SELECT * FROM autoloader.gold.customer_scd where  order_id
ORDER BY order_id, start_date; 

In [0]:
%sql
SELECT order_id, count(*) 
FROM autoloader.gold.customer_scd
WHERE is_current = 'Y'
GROUP BY order_id
HAVING count(*) != 1;

In [0]:
%sql
-- Step 8: Final Validation
SELECT 
  count(*) as total_records,
  count(DISTINCT order_id) as unique_orders,
  min(ingestion_timestamp) as first_ingested,
  max(ingestion_timestamp) as last_ingested
FROM autoloader.bronze.sales;